In [ ]:
import sys

print("Python executable:", sys.executable)
print("Python version:", sys.version)

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# System utilities
import os
import warnings

# Display settings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Visualization settings
sns.set_theme(style="whitegrid")

print("All libraries imported successfully!")

In [ ]:
# Dataset path
DATA_PATH = "data/P4-LUCAT Research Data pub.csv"

# Load dataset
df = pd.read_csv(
    DATA_PATH,
    sep=";",
    encoding="utf-8-sig"
)

print("Dataset loaded successfully!")

In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

In [ ]:
display(df.head())

In [ ]:
print("Total columns:", len(df.columns))

for i, column in enumerate(df.columns, start=1):
    print(i, column)

In [ ]:
# Dataset information
df.info()

In [ ]:
# Calculate missing values
missing_values = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": (df.isnull().sum() / len(df)) * 100
})

# Show columns with missing values
missing_values = missing_values[
    missing_values["Missing Count"] > 0
].sort_values(by="Missing Percentage", ascending=False)

display(missing_values.head(30).round(2))

In [ ]:
TARGET = "response_pharmtreat1_CUI"

print("Target variable:", TARGET)
print("\nMissing target values:", df[TARGET].isnull().sum())

print("\nTarget categories:")
print(df[TARGET].value_counts(dropna=False))

In [ ]:
plt.figure(figsize=(10, 5))

sns.countplot(
    data=df,
    x=TARGET,
    order=df[TARGET].value_counts().index
)

plt.title("Distribution of First Treatment Response Categories")
plt.xlabel("Treatment Response")
plt.ylabel("Number of Patients")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Preserve original dataset
df_original = df.copy()

# Create modelling dataset with recorded target values
df_model = df[
    df["response_pharmtreat1_CUI"].notna()
].copy()

print("Original dataset shape:", df_original.shape)
print("Modelling dataset shape:", df_model.shape)

print(
    "Excluded records:",
    len(df_original) - len(df_model)
)

In [ ]:
# Response category distribution
response_distribution = (
    df_model["response_pharmtreat1_CUI"]
    .value_counts()
    .reset_index()
)

response_distribution.columns = [
    "Response Category",
    "Patient Count"
]

response_distribution["Percentage"] = (
    response_distribution["Patient Count"]
    / len(df_model) * 100
)

display(response_distribution.round(2))

In [ ]:
print(
    "Duplicate rows:",
    df_model.duplicated().sum()
)

In [ ]:
response_distribution.to_csv(
    "outputs/results/response_distribution.csv",
    index=False
)

print("Response distribution saved successfully!")

In [ ]:
# Define feature groups for initial leakage screening

feature_groups = {
    "Demographic and clinical": [
        "Gender", "AgeGroup", "race", "smokinghabit",
        "backgroundind", "numfamlung", "numfamother",
        "orgfam", "comorind", "comorbidity",
        "comorbidity_CUI", "diagnosisdate"
    ],

    "Diagnosis and cancer characteristics": [
        "inistage", "inistage_CUI", "version",
        "histology"
    ],

    "Molecular information": [
        col for col in df_model.columns
        if col.startswith(("molmar", "result"))
        or col == "positividad_pdl1"
    ],

    "Treatment information": [
        col for col in df_model.columns
        if col.startswith(("phartreat", "schema", "numcycles"))
    ],

    "Treatment toxicity": [
        col for col in df_model.columns
        if "toxicity" in col.lower()
        or col.startswith("indtox")
    ],

    "Progression and survival": [
        col for col in df_model.columns
        if col.startswith((
            "relprog", "fecrelprog", "progtype",
            "deceased", "decdate", "sur"
        ))
    ],

    "Radiotherapy": [
        col for col in df_model.columns
        if col.startswith(("rtthreatment", "rtstartdate",
                           "rtenddate", "response_RT",
                           "indtoxrt", "toxicity_RT"))
    ],

    "Surgery and procedures": [
        col for col in df_model.columns
        if col.startswith(("surtype", "surclass", "surdate",
                           "procedure", "perimor"))
    ]
}

for group, columns in feature_groups.items():
    print(f"\n{'='*60}")
    print(group)
    print(f"Number of columns: {len(columns)}")
    print(columns)

In [ ]:
# Variables that are clearly outcomes or recorded after treatment

high_risk_keywords = [
    "response_",
    "toxicity",
    "indtox",
    "deceased",
    "decdate",
    "relprog",
    "fecrelprog",
    "progtype",
    "surdate",
    "phartreatenddate",
    "rtenddate"
]

high_risk_columns = [
    col for col in df_model.columns
    if any(keyword in col.lower()
           for keyword in high_risk_keywords)
]

print("Potential post-treatment / outcome variables:")
for col in high_risk_columns:
    print("-", col)

print("\nTotal potential high-risk columns:", len(high_risk_columns))

In [ ]:
first_treatment_columns = [
    col for col in df_model.columns
    if any(term in col.lower() for term in [
        "phartreat1",
        "phartreatstartdate1",
        "phartreatenddate1",
        "numcycles1",
        "schema1",
        "response_pharmtreat1",
        "toxicity_pharmtreat1"
    ])
]

display(df_model[first_treatment_columns].head(10))

In [ ]:
# Candidate baseline demographic features
demographic_features = [
    "Gender",
    "AgeGroup",
    "race",
    "smokinghabit",
    "backgroundind",
    "numfamlung",
    "numfamother",
    "orgfam"
]

# Candidate clinical features
clinical_features = [
    "comorind",
    "comorbidity",
    "inistage",
    "version",
    "histology"
]

# Candidate molecular features
molecular_features = [
    "molmarind",
    "molmar1", "molmar2", "molmar3", "molmar4",
    "molmar5", "molmar6", "molmar7", "molmar8",
    "result1", "result2", "result3", "result4",
    "result5", "result6", "result7", "result8",
    "positividad_pdl1"
]

# Planned first treatment features
treatment_features = [
    "phartreat1",
    "schema1"
]

# Keep only columns that exist
candidate_features = (
    demographic_features
    + clinical_features
    + molecular_features
    + treatment_features
)

candidate_features = [
    col for col in candidate_features
    if col in df_model.columns
]

print("Total candidate features:", len(candidate_features))
print(candidate_features)

In [ ]:
feature_availability = pd.DataFrame({
    "Feature": candidate_features,
    "Available Records": [
        df_model[col].notna().sum()
        for col in candidate_features
    ],
    "Missing Records": [
        df_model[col].isna().sum()
        for col in candidate_features
    ],
    "Availability (%)": [
        df_model[col].notna().mean() * 100
        for col in candidate_features
    ],
    "Unique Values": [
        df_model[col].nunique()
        for col in candidate_features
    ]
})

feature_availability = feature_availability.sort_values(
    by="Availability (%)",
    ascending=False
)

display(feature_availability.round(2))

In [ ]:
# Calculate missing values
missing_values = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": (df.isnull().sum() / len(df)) * 100
})

# Show columns with missing values
missing_values = missing_values[
    missing_values["Missing Count"] > 0
].sort_values(by="Missing Percentage", ascending=False)

display(missing_values.head(30).round(2))

In [ ]:
molecular_inspection = [
    col for col in candidate_features
    if col in molecular_features
]

display(df_model[molecular_inspection].head(10))

In [ ]:
treatment_columns = [
    "diagnosisdate",
    "phartreat1",
    "phartreatstartdate1",
    "phartreatenddate1",
    "numcycles1",
    "schema1",
    "toxicity_pharmtreat1",
    "response_pharmtreat1_CUI"
]

display(
    df_model[treatment_columns].head(20)
)

In [ ]:
print("First treatment categories:")

display(
    df_model["phartreat1"]
    .value_counts(dropna=False)
    .to_frame("Patient Count")
)

print("\nFirst treatment regimens:")

display(
    df_model["schema1"]
    .value_counts(dropna=False)
    .head(20)
    .to_frame("Patient Count")
)

In [ ]:
date_check = df_model[
    ["diagnosisdate", "phartreatstartdate1"]
].copy()

date_check["diagnosisdate"] = pd.to_datetime(
    date_check["diagnosisdate"],
    format="%d/%m/%Y",
    errors="coerce"
)

date_check["phartreatstartdate1"] = pd.to_datetime(
    date_check["phartreatstartdate1"],
    format="%d/%m/%Y",
    errors="coerce"
)

date_check["days_to_treatment"] = (
    date_check["phartreatstartdate1"]
    - date_check["diagnosisdate"]
).dt.days

display(date_check.head(15))

print("\nInterval statistics:")
display(date_check["days_to_treatment"].describe())

print(
    "\nNegative intervals:",
    (date_check["days_to_treatment"] < 0).sum()
)

In [ ]:
print("Molecular marker 1:")
display(df_model["molmar1"].value_counts(dropna=False))

print("\nMolecular test result 1:")
display(df_model["result1"].value_counts(dropna=False))

print("\nMolecular testing indicator:")
display(df_model["molmarind"].value_counts(dropna=False))

In [ ]:
# Identify negative diagnosis-to-treatment intervals
negative_intervals = date_check[
    date_check["days_to_treatment"] < 0
]

print("Negative interval records:")
display(negative_intervals)

# Identify unusually long intervals
long_intervals = date_check[
    date_check["days_to_treatment"] > 365
]

print("Intervals greater than 365 days:", len(long_intervals))
display(long_intervals.head(20))

In [ ]:
import ast

def extract_drugs(value):
    if pd.isna(value):
        return []

    try:
        parsed = ast.literal_eval(value)

        if isinstance(parsed, list):
            return [str(drug).strip() for drug in parsed]

        return [str(parsed).strip()]

    except (ValueError, SyntaxError):
        return []

df_model["drugs_list"] = df_model["schema1"].apply(extract_drugs)

print("Extracted treatment combinations:")
display(df_model[["schema1", "drugs_list"]].head(15))

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()

drug_encoded = mlb.fit_transform(df_model["drugs_list"])

drug_df = pd.DataFrame(
    drug_encoded,
    columns=[f"drug_{drug}" for drug in mlb.classes_],
    index=df_model.index
)

print("Number of individual drugs:", len(mlb.classes_))

display(drug_df.head())

In [ ]:
drug_frequency = (
    drug_df.sum()
    .sort_values(ascending=False)
    .reset_index()
)

drug_frequency.columns = ["Drug", "Patient Count"]

display(drug_frequency)

In [ ]:
rare_drugs = drug_frequency[
    drug_frequency["Patient Count"] < 10
]

display(rare_drugs)

In [ ]:
trial_entries = [
    drug for drug in mlb.classes_
    if "placebo" in drug.lower()
    or "vs" in drug.lower()
    or drug.lower() == "otro"
]

print("Trial-related or special entries:")

for drug in trial_entries:
    print("-", drug)

In [ ]:
trial_keywords = ["placebo", "otro"]

trial_mask = df_model["schema1"].apply(
    lambda value: (
        isinstance(value, str)
        and any(keyword in value.lower() for keyword in trial_keywords)
    )
)

trial_records = df_model.loc[
    trial_mask,
    ["phartreat1", "schema1", "response_pharmtreat1_CUI"]
]

display(trial_records)

In [ ]:
print("Number of records:", len(trial_records))

print("\nTreatment category distribution:")
display(trial_records["phartreat1"].value_counts(dropna=False))

print("\nOriginal regimen distribution:")
display(trial_records["schema1"].value_counts(dropna=False))

In [ ]:
import ast
import pandas as pd

def clean_drug_list(value):
    if pd.isna(value):
        return []

    try:
        drugs = ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return []

    if not isinstance(drugs, list):
        return []

    cleaned = []

    for drug in drugs:
        drug_lower = drug.lower()

        if "vs placebo" in drug_lower:
            continue

        if drug_lower == "otro":
            continue

        cleaned.append(drug.strip())

    return cleaned


df_model["clean_drugs_list"] = (
    df_model["schema1"].apply(clean_drug_list)
)

display(
    df_model[
        ["schema1", "clean_drugs_list"]
    ].head(15)
)

In [ ]:
print("Original drug entries:", sum(len(x) for x in df_model["drugs_list"]))
print("Cleaned drug entries:", sum(len(x) for x in df_model["clean_drugs_list"]))

print("\nRecords with missing regimen:")
print(df_model["schema1"].isna().sum())

print("\nRecords with no named drug after cleaning:")
print((df_model["clean_drugs_list"].apply(len) == 0).sum())

In [ ]:
empty_drug_records = df_model[
    df_model["clean_drugs_list"].apply(len) == 0
]

display(
    empty_drug_records[
        [
            "phartreat1",
            "schema1",
            "response_pharmtreat1_CUI"
        ]
    ]
)

print("Total records:", len(empty_drug_records))

print("\nMissing schema1:")
print(empty_drug_records["schema1"].isna().sum())

print("\nNon-missing schema1:")
print(empty_drug_records["schema1"].notna().sum())

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

drug_mlb = MultiLabelBinarizer()

clean_drug_encoded = drug_mlb.fit_transform(
    df_model["clean_drugs_list"]
)

clean_drug_df = pd.DataFrame(
    clean_drug_encoded,
    columns=[
        f"drug_{drug}"
        for drug in drug_mlb.classes_
    ],
    index=df_model.index
)

# Preserve missing or ambiguous regimen information
df_model["regimen_missing_or_ambiguous"] = (
    df_model["schema1"].isna()
    | df_model["clean_drugs_list"].apply(len).eq(0)
).astype(int)

print("Clean drug feature shape:", clean_drug_df.shape)

print("\nRegimen information status:")
display(
    df_model["regimen_missing_or_ambiguous"].value_counts()
)

display(clean_drug_df.head())

In [ ]:
print("Drug feature shape:", clean_drug_df.shape)

print(
    "Total encoded drug occurrences:",
    clean_drug_df.sum().sum()
)

print(
    "Number of unique values in drug indicators:",
    sorted(clean_drug_df.stack().unique())
)

print(
    "Missing/ambiguous regimen count:",
    df_model["regimen_missing_or_ambiguous"].sum()
)

print(
    "Index alignment:",
    clean_drug_df.index.equals(df_model.index)
)

print(
    "Target alignment:",
    df_model["response_pharmtreat1_CUI"].notna().all()
)

In [ ]:
import matplotlib.pyplot as plt
import os

# Calculate patient counts for each drug
drug_counts = (
    clean_drug_df.sum()
    .sort_values(ascending=False)
)

# Select top 15 drugs
top_15_drugs = drug_counts.head(15).sort_values()

# Create output directory
os.makedirs("outputs/figures", exist_ok=True)
os.makedirs("outputs/results", exist_ok=True)

# Plot
plt.figure(figsize=(11, 7))

top_15_drugs.plot(kind="barh")

plt.title("Top 15 Most Frequently Recorded Drugs")
plt.xlabel("Number of Patients")
plt.ylabel("Drug")
plt.tight_layout()

# Save graph
plt.savefig(
    "outputs/figures/top_15_drugs.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# Save frequency results
drug_counts.sort_values(ascending=False).to_csv(
    "outputs/results/clean_drug_frequency.csv",
    header=["Patient Count"]
)

print("Drug distribution graph saved successfully.")

In [ ]:
treatment_counts = (
    df_model["phartreat1"]
    .value_counts()
    .sort_values(ascending=True)
)

plt.figure(figsize=(11, 7))

bars = plt.barh(
    treatment_counts.index,
    treatment_counts.values
)

plt.title("Distribution of First Treatment Categories")
plt.xlabel("Number of Patients")
plt.ylabel("First Treatment Category")

for bar in bars:
    plt.text(
        bar.get_width() + 2,
        bar.get_y() + bar.get_height() / 2,
        int(bar.get_width()),
        va="center"
    )

plt.tight_layout()

plt.savefig(
    "outputs/figures/first_treatment_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

treatment_counts.sort_values(ascending=False).to_csv(
    "outputs/results/first_treatment_distribution.csv",
    header=["Patient Count"]
)

print("Treatment distribution graph and results saved.")

In [ ]:
demographic_features = [
    "Gender",
    "AgeGroup",
    "race",
    "smokinghabit"
]

for feature in demographic_features:
    print(f"\n{'='*50}")
    print(f"{feature} distribution")
    print("="*50)

    display(
        df_model[feature]
        .value_counts(dropna=False)
        .rename_axis(feature)
        .reset_index(name="Patient Count")
    )

In [ ]:
import matplotlib.pyplot as plt
import os

demographic_features = [
    "Gender",
    "AgeGroup",
    "race",
    "smokinghabit"
]

os.makedirs("outputs/figures", exist_ok=True)
os.makedirs("outputs/results", exist_ok=True)

for feature in demographic_features:

    counts = df_model[feature].value_counts(dropna=False)

    plt.figure(figsize=(9, 5))

    bars = plt.bar(
        counts.index.astype(str),
        counts.values
    )

    plt.title(f"Distribution of {feature}")
    plt.xlabel(feature)
    plt.ylabel("Number of Patients")

    plt.xticks(rotation=25, ha="right")

    for bar in bars:
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 3,
            int(bar.get_height()),
            ha="center"
        )

    plt.tight_layout()

    filename = feature.lower() + "_distribution.png"

    plt.savefig(
        f"outputs/figures/{filename}",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()

    counts.rename("Patient Count").to_csv(
        f"outputs/results/{feature.lower()}_distribution.csv"
    )

print("All four demographic graphs and CSV files saved.")

In [ ]:
clinical_features = [
    "backgroundind",
    "comorind",
    "comorbidity",
    "inistage",
    "histology"
]

for feature in clinical_features:
    print(f"\n{'='*55}")
    print(f"{feature} distribution")
    print("="*55)

    print("Available records:", df_model[feature].notna().sum())
    print("Missing records:", df_model[feature].isna().sum())
    print("Unique categories:", df_model[feature].nunique())

    display(
        df_model[feature]
        .value_counts(dropna=False)
        .rename_axis(feature)
        .reset_index(name="Patient Count")
    )

In [ ]:
clinical_features = [
    "backgroundind",
    "comorind",
    "comorbidity",
    "inistage",
    "histology"
]

os.makedirs("outputs/figures", exist_ok=True)
os.makedirs("outputs/results", exist_ok=True)

for feature in clinical_features:

    counts = df_model[feature].value_counts(
        dropna=False
    )

    # Convert missing values to readable labels
    counts.index = [
        "Missing" if pd.isna(value) else str(value)
        for value in counts.index
    ]

    plt.figure(figsize=(11, 6))

    bars = plt.barh(
        counts.index[::-1],
        counts.values[::-1]
    )

    plt.title(f"Distribution of {feature}")
    plt.xlabel("Number of Patients")
    plt.ylabel(feature)

    for bar in bars:
        plt.text(
            bar.get_width() + 2,
            bar.get_y() + bar.get_height() / 2,
            int(bar.get_width()),
            va="center"
        )

    plt.tight_layout()

    plt.savefig(
        f"outputs/figures/{feature}_distribution.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()

    counts.rename("Patient Count").to_csv(
        f"outputs/results/{feature}_distribution.csv"
    )

print("All five clinical distribution graphs and CSV files saved.")

In [ ]:
# Calculate missing-value statistics
missing_summary = pd.DataFrame({
    "Missing Count": df_model.isnull().sum(),
    "Missing Percentage": (
        df_model.isnull().mean() * 100
    )
})

# Keep columns that contain missing values
missing_summary = (
    missing_summary[missing_summary["Missing Count"] > 0]
    .sort_values("Missing Percentage", ascending=False)
)

# Display top 25 columns
top_missing = missing_summary.head(25)

display(top_missing)

In [ ]:
candidate_features = [
    "Gender",
    "AgeGroup",
    "race",
    "smokinghabit",
    "backgroundind",
    "numfamlung",
    "numfamother",
    "comorind",
    "comorbidity",
    "inistage",
    "histology",
    "phartreat1",
    "schema1"
]

candidate_missing = pd.DataFrame({
    "Available Count": df_model[candidate_features].notna().sum(),
    "Missing Count": df_model[candidate_features].isna().sum(),
    "Missing Percentage": (
        df_model[candidate_features].isna().mean() * 100
    )
})

display(candidate_missing)

In [ ]:
features_to_check = [
    "numfamlung",
    "numfamother",
    "comorbidity"
]

for feature in features_to_check:

    print(f"\n{'='*50}")
    print(f"Missingness analysis: {feature}")
    print("="*50)

    missing_group = df_model[feature].isna()

    print("Missing records:", missing_group.sum())
    print("Available records:", (~missing_group).sum())

    print("\nComorbidity indicator:")
    display(
        pd.crosstab(
            missing_group,
            df_model["comorind"],
            margins=True
        )
    )

In [ ]:
family_features = ["numfamlung", "numfamother"]

for feature in family_features:

    print(f"\n{'='*50}")
    print(f"{feature} analysis")
    print("="*50)

    print("Data type:", df_model[feature].dtype)

    print("\nDescriptive statistics:")
    display(df_model[feature].describe())

    print("\nUnique values:")
    print(sorted(df_model[feature].dropna().unique()))

    print("\nValue frequencies:")
    display(
        df_model[feature]
        .value_counts(dropna=False)
        .sort_index()
    )

In [ ]:
family_features = ["numfamlung", "numfamother"]

for feature in family_features:

    plt.figure(figsize=(8, 5))

    values = df_model[feature].dropna()

    bins = range(int(values.min()), int(values.max()) + 2)

    plt.hist(
        values,
        bins=bins,
        edgecolor="black",
        align="left"
    )

    plt.title(f"Distribution of {feature}")
    plt.xlabel("Number of Family Members")
    plt.ylabel("Number of Patients")

    plt.xticks(range(int(values.min()), int(values.max()) + 1))

    plt.tight_layout()

    plt.savefig(
        f"outputs/figures/{feature}_distribution.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()

print("Family-history distribution graphs saved.")

In [ ]:
features_to_check = [
    "Gender",
    "AgeGroup",
    "smokinghabit",
    "backgroundind",
    "comorind",
    "inistage",
    "histology",
    "phartreat1"
]

target = "response_pharmtreat1_CUI"

for feature in features_to_check:
    print(f"\n{'='*60}")
    print(f"FEATURE: {feature}")
    print("="*60)

    result = pd.crosstab(
        df_model[feature],
        df_model[target],
        normalize="index"
    ) * 100

    print(result.round(1))

In [ ]:
import os

os.makedirs("outputs/results", exist_ok=True)

with pd.ExcelWriter(
    "outputs/results/feature_target_relationships.xlsx"
) as writer:

    for feature in features_to_check:

        result = pd.crosstab(
            df_model[feature],
            df_model[target],
            normalize="index"
        ) * 100

        result.round(2).to_excel(
            writer,
            sheet_name=feature[:31]
        )

print("All feature-target relationship tables saved successfully!")

In [ ]:
# Sample counts for each feature category

for feature in features_to_check:

    print("\n" + "=" * 60)
    print("FEATURE:", feature)
    print("=" * 60)

    counts = pd.crosstab(
        df_model[feature],
        df_model[target]
    )

    counts["Total"] = counts.sum(axis=1)

    print(counts.to_string())

In [ ]:
# Identify categories with small sample sizes

summary = []

for feature in features_to_check:

    counts = df_model[feature].value_counts(dropna=False)

    for category, count in counts.items():
        summary.append({
            "Feature": feature,
            "Category": str(category),
            "Patient_Count": count,
            "Percentage": round(count / len(df_model) * 100, 2),
            "Rare_Category": "Yes" if count < 20 else "No"
        })

category_summary = pd.DataFrame(summary)

print("RARE CATEGORIES (fewer than 20 patients):")
print(
    category_summary[
        category_summary["Rare_Category"] == "Yes"
    ].to_string(index=False)
)

category_summary.to_csv(
    "outputs/results/feature_category_sample_counts.csv",
    index=False
)

print("\nComplete category counts saved successfully!")

In [ ]:
# Inspect treatment-related columns and their availability

treatment_columns = [
    col for col in df_model.columns
    if any(term in col.lower() for term in [
        "phartreat", "schema", "startdate",
        "enddate", "cycle", "response"
    ])
]

for col in treatment_columns:
    print(f"{col}: {df_model[col].notna().sum()} non-missing values")

In [ ]:
for col in df_model.columns:
    if any(term in col.lower() for term in [
        "phartreat", "schema", "numcycles", "response"
    ]):
        print(col)

In [ ]:
# PHASE 2: Final temporal leakage audit

import pandas as pd

date_cols = ["diagnosisdate", "phartreatstartdate1",
             "phartreatenddate1"]

for col in date_cols:
    df_model[col] = pd.to_datetime(
        df_model[col],
        format="%d/%m/%Y",
        errors="coerce"
    )

audit = pd.DataFrame({
    "Diagnosis_Date": df_model["diagnosisdate"],
    "Treatment_Start": df_model["phartreatstartdate1"],
    "Treatment_End": df_model["phartreatenddate1"],
    "First_Treatment": df_model["phartreat1"],
    "Drug_Regimen": df_model["schema1"],
    "Target": df_model["response_pharmtreat1_CUI"]
})

audit["Diagnosis_to_Treatment_Days"] = (
    audit["Treatment_Start"] - audit["Diagnosis_Date"]
).dt.days

print("TEMPORAL LEAKAGE AUDIT")
print("-" * 55)

print("Patients:", len(audit))
print("Missing diagnosis dates:",
      audit["Diagnosis_Date"].isna().sum())
print("Missing treatment start dates:",
      audit["Treatment_Start"].isna().sum())
print("Missing treatment end dates:",
      audit["Treatment_End"].isna().sum())
print("Missing first treatment:",
      audit["First_Treatment"].isna().sum())
print("Missing drug regimen:",
      audit["Drug_Regimen"].isna().sum())

print("\nDiagnosis-to-treatment interval:")
print(audit["Diagnosis_to_Treatment_Days"].describe())

print("\nNegative intervals:",
      (audit["Diagnosis_to_Treatment_Days"] < 0).sum())

print("Intervals greater than 365 days:",
      (audit["Diagnosis_to_Treatment_Days"] > 365).sum())

print("\nTreatment start after treatment end:",
      (audit["Treatment_Start"] > audit["Treatment_End"]).sum())

audit.to_csv(
    "outputs/results/temporal_leakage_audit.csv",
    index=False
)

print("\nAudit saved successfully!")

In [ ]:
# Investigate unusual diagnosis-to-treatment intervals

anomalies = audit[
    (audit["Diagnosis_to_Treatment_Days"] < 0) |
    (audit["Diagnosis_to_Treatment_Days"] > 365)
].copy()

print("Number of unusual records:", len(anomalies))

print("\nUnusual interval records:")

print(
    anomalies[
        [
            "Diagnosis_Date",
            "Treatment_Start",
            "Treatment_End",
            "Diagnosis_to_Treatment_Days",
            "First_Treatment"
        ]
    ].sort_values(
        "Diagnosis_to_Treatment_Days"
    ).to_string()
)

print("\nResponse distribution in unusual records:")

print(
    anomalies["Target"].value_counts()
)

In [ ]:
# Save all unusual date records

anomalies.to_csv(
    "outputs/results/unusual_date_intervals.csv",
    index=True
)

print("Unusual records saved:", len(anomalies))
print("File: outputs/results/unusual_date_intervals.csv")

In [ ]:
# Final EDA summary

eda_summary = """
PROJECT: Explainable ML for Personalized Treatment
Response Prediction in Lung Cancer

DATASET:
P4-LUCAT

ORIGINAL DATASET:
1242 rows, 174 columns

MODELLING DATASET:
888 patients with recorded treatment response

EXCLUDED:
354 patients with missing target values

TARGET:
response_pharmtreat1_CUI

NUMBER OF RESPONSE CLASSES:
6

MAJOR FINDINGS:
1. Male patients: 617
2. Female patients: 271
3. Most patients belong to the 64-90 age group.
4. Most patients are Caucasian.
5. Former and active smokers form the majority.
6. Adenocarcinoma is the most frequent histology.
7. RP is the largest response category.
8. The target classes are imbalanced.
9. Family-history variables contain missing values.
10. Several categorical features contain rare categories.

TEMPORAL AUDIT:
Patients examined: 888
Missing first treatment start dates: 18
Missing first treatment end dates: 28
Negative diagnosis-treatment intervals: 2
Intervals exceeding 365 days: 29

DATA QUALITY:
31 unusual date intervals identified and exported.

MODELLING CONSIDERATIONS:
1. Preserve all six response categories.
2. Do not automatically remove rare categories.
3. Use training-only imputation and encoding.
4. Exclude post-treatment information.
5. Investigate timing of first-treatment variables.
6. Evaluate class imbalance during modelling.

NEXT PHASE:
Feature Selection and Leakage Prevention.
"""

with open(
    "outputs/results/final_eda_summary.txt",
    "w"
) as file:
    file.write(eda_summary)

print("Final EDA summary saved successfully!")

In [ ]:
import os
import json
import pandas as pd

# Target variable
target = "response_pharmtreat1_CUI"

# Model A: Clinical-only features
clinical_features = [
    "Gender",
    "AgeGroup",
    "race",
    "smokinghabit",
    "backgroundind",
    "comorind",
    "comorbidity",
    "numfamlung",
    "numfamother",
    "inistage",
    "histology"
]

# Model B: Clinical + first-treatment information
treatment_features = clinical_features + [
    "phartreat1",
    "schema1"
]

# Verify all selected columns exist
for feature in treatment_features + [target]:
    if feature not in df_model.columns:
        raise ValueError(f"Missing column: {feature}")

# Verify target
assert df_model[target].notna().all()

# Save feature selection
feature_plan = {
    "target": target,
    "model_A_clinical_only": clinical_features,
    "model_B_treatment_informed": treatment_features
}

os.makedirs("outputs/results", exist_ok=True)

with open(
    "outputs/results/feature_selection_plan.json",
    "w"
) as f:
    json.dump(feature_plan, f, indent=4)

print("FEATURE SELECTION PLAN")
print("-" * 50)

print("Target:", target)

print("\nModel A - Clinical-only")
for feature in clinical_features:
    print(feature)

print("\nModel B - Treatment-informed")
for feature in treatment_features:
    print(feature)

print("\nModel A feature count:", len(clinical_features))
print("Model B feature count:", len(treatment_features))

print("\nFeature selection plan saved successfully!")

In [ ]:
# Feature audit for both experimental models

import pandas as pd

for model_name, features in [
    ("Model A - Clinical-only", clinical_features),
    ("Model B - Treatment-informed", treatment_features)
]:

    print("\n" + "=" * 65)
    print(model_name)
    print("=" * 65)

    audit_table = pd.DataFrame({
        "Data_Type": df_model[features].dtypes.astype(str),
        "Missing_Count": df_model[features].isna().sum(),
        "Missing_Percentage": (
            df_model[features].isna().mean() * 100
        ).round(2),
        "Unique_Values": df_model[features].nunique()
    })

    print(audit_table.to_string())

    audit_table.to_csv(
        f"outputs/results/{model_name.split()[1].lower()}_feature_audit.csv"
    )

print("\nFeature audits saved successfully!")

In [ ]:
# Investigate comorbidity missingness

print("Comorbidity missingness vs comorbidity indicator")
print("-" * 55)

print(
    pd.crosstab(
        df_model["comorind"],
        df_model["comorbidity"].isna(),
        margins=True
    )
)

print("\nComorbidity values:")
print(df_model["comorbidity"].value_counts(dropna=False))

In [ ]:
clinical_features = [
    "Gender",
    "AgeGroup",
    "race",
    "smokinghabit",
    "backgroundind",
    "comorind",
    "numfamlung",
    "numfamother",
    "inistage",
    "histology"
]

In [ ]:
treatment_features = clinical_features + [
    "phartreat1",
    "schema1"
]

In [ ]:
import json

# Final primary feature sets
clinical_features = [
    "Gender",
    "AgeGroup",
    "race",
    "smokinghabit",
    "backgroundind",
    "comorind",
    "numfamlung",
    "numfamother",
    "inistage",
    "histology"
]

treatment_features = clinical_features + [
    "phartreat1",
    "schema1"
]

feature_plan = {
    "target": "response_pharmtreat1_CUI",
    "model_A_clinical_only": clinical_features,
    "model_B_treatment_informed": treatment_features,
    "excluded_from_primary": [
        "comorbidity",
        "phartreat2_to_phartreat8",
        "treatment_dates",
        "numcycles1",
        "other_response_variables",
        "toxicity_variables"
    ]
}

with open(
    "outputs/results/feature_selection_plan.json",
    "w"
) as f:
    json.dump(feature_plan, f, indent=4)

print("FINAL FEATURE PLAN SAVED")
print("Model A:", len(clinical_features), "features")
print("Model B:", len(treatment_features), "features")
print("Excluded: comorbidity due to structural missingness")